# Pertemuan 4 — Mendiagnosis Kualitas Data
### Workbook belajar mandiri

**Exploratory Data Analysis (KMTI21133)**  
**Sub-CPMK4:** Mahasiswa mampu mendiagnosis masalah kualitas data: *missing values*, duplikasi, inkonsistensi, dan kesalahan tipe data (C4).

---

> **Dosen berhalangan hadir pada pertemuan 4 dan 5.** Kelas didampingi asisten, dan workbook ini dirancang agar
> dapat Anda selesaikan sendiri — di kelas bersama asisten, atau sepenuhnya mandiri di luar kelas.

### Cara memakai workbook ini

| | |
|---|---|
| **Perkiraan waktu** | 150 menit. Bagian 1–5 sekitar 75 menit, Bagian 6–9 sekitar 50 menit, Bagian 10 sekitar 25 menit. |
| **Sel penjelasan** | Dibaca. Tidak ada yang perlu dijalankan. |
| **Sel CONTOH** | Dijalankan apa adanya, lalu amati keluarannya. |
| **Sel TUGAS** | Anda isi sendiri, lalu jalankan sel `cek(...)` di bawahnya. *(Pada versi ini semua tugas sudah terisi sebagai kunci jawaban.)* |
| **Umpan balik** | Sel `cek(...)` akan menjawab **BENAR** atau **BELUM** beserta petunjuk. |
| **Kalau mentok** | Lihat Bagian *Kalau Anda mentok* di akhir workbook sebelum bertanya. |

### Yang Anda hasilkan di akhir sesi

1. **Profil kualitas data** untuk dataset latihan (Bagian 9).
2. **Profil kualitas data awal untuk dataset proyek tim** (Bagian 10) — ini bagian pertama dari **M2** yang dikumpulkan pada pertemuan 5.

> **Jangan memperbaiki apa pun hari ini.** Pertemuan 4 hanya mendiagnosis. Perbaikan dikerjakan pada pertemuan 5.
> Memperbaiki sebelum mendiagnosis adalah cara tercepat menghilangkan bukti.

---
## Bagian 0 — Persiapan

Dataset: **`genshin_impact_clean.csv`** — 103 karakter Genshin Impact: atribut karakter (senjata, elemen, wilayah, peran)
dan statistik jumlah *pull* per tingkat konstelasi. Nama karakter sudah dianonimkan menjadi kode heksadesimal.

> **Catatan penting tentang dataset ini.** Namanya `clean`, artinya sebagian pembersihan sudah dilakukan oleh pembuatnya
> (tidak ada sel kosong, tidak ada kolom tanggal). Jangan percaya label itu begitu saja: tugas Anda justru memeriksa
> apakah data ini **benar-benar** bersih. Hasilnya sering berupa *"nol temuan"* di satu dimensi dan temuan halus di dimensi lain.
> Mengetahui bahwa sebuah pemeriksaan **lolos** juga merupakan hasil diagnosis.

In [ ]:
import pandas as pd
import numpy as np
import hashlib, os

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

FILE = 'genshin_impact_clean.csv'
if not os.path.exists(FILE):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        raise FileNotFoundError('Letakkan CSV di folder yang sama dengan notebook ini.')

df = pd.read_csv(FILE)
print('Berhasil dimuat:', df.shape)

Berhasil dimuat: (103, 24)


### Alat pemeriksa jawaban
Jalankan sel berikut satu kali. Setelah itu Anda bisa memanggil `cek('T1_baris', jawaban_saya)` kapan saja.

In [ ]:
_K = {
    'T1_baris': '454f63ac30',
    'T1_kolom': 'c2356069e9',
    'T2_kosong': '5feceb66ff',
    'T2_kolom': '5feceb66ff',
    'T3_placeholder': '5feceb66ff',
    'T3_nol': '5feceb66ff',
    'T4_mar': '43b87f618c',
    'T4_bias': '4e91f1ce21',
    'T5_dup': '5feceb66ff',
    'T5_dupnama': '5feceb66ff',
    'T5_nyaris': '5feceb66ff',
    'T6_kat': '19581e27de',
    'T6_katbersih': '19581e27de',
    'T6_jarang': 'd4735e3a26',
    'T9_nama': '454f63ac30',
    'T9_strip': '454f63ac30',
    'T9_hex': '454f63ac30',
    'T7_peran': 'ef2d127de3',
    'T7_kombinasi': '6b51d431df',
    'T7_biner': '6b86b273ff',
    'T12_bulan': '4b227777d4',
    'T12_rusak': '4b227777d4',
    'T8_rarity': '5feceb66ff',
    'T8_rate': '5feceb66ff',
    'T10_bisnis': '19581e27de',
    'T10_selisih': '4e07408562',
    'T10_turunan': '5feceb66ff'
}

_P = {
    'T1_baris': 'Pakai df.shape[0] atau len(df).',
    'T1_kolom': 'Pakai df.shape[1].',
    'T2_kosong': 'df.isna().sum() memberi per kolom; jumlahkan lagi dengan .sum().',
    'T2_kolom': 'Hitung berapa kolom yang nilai isna().sum()-nya lebih dari 0.',
    'T3_placeholder': 'Pakai .isin([-999, -1, 9999, 99999, 999999]) pada kolom numerik, lalu jumlahkan dua kali.',
    'T3_nol': 'Buang is_standard_banner (kolom biner, nol itu wajar) lalu hitung sel numerik yang == 0.',
    'T4_mar': 'Pada df_mar, hitung isna().mean()*100 untuk baris rarity 4. Bulatkan 1 desimal.',
    'T4_bias': 'Rata-rata pulled_count di df_mnar dibagi rata-rata aslinya di df, kali 100. Bulatkan 1 desimal.',
    'T5_dup': 'df.duplicated().sum() menghitung baris yang SELURUH kolomnya sama.',
    'T5_dupnama': "Pakai df['name'].duplicated(keep=False).sum().",
    'T5_nyaris': "Buang kolom name, lalu duplicated(keep=False).sum(): baris yang sama persis kecuali namanya.",
    'T6_kat': 'Hitung nunique() pada kolom region apa adanya.',
    'T6_katbersih': 'Rapikan dengan .str.strip().str.title() lalu hitung nunique(). Apakah jumlahnya berubah?',
    'T6_jarang': 'Hitung kategori region yang value_counts()-nya kurang dari atau sama dengan 2.',
    'T9_nama': 'nunique() pada name apa adanya.',
    'T9_strip': 'nunique() pada name setelah .str.strip().',
    'T9_hex': "Pakai .str.fullmatch(r'[0-9a-f]{12}') lalu jumlahkan.",
    'T7_peran': "Pisahkan roles dengan .str.split(',').explode() lalu nunique().",
    'T7_kombinasi': 'nunique() pada kolom roles apa adanya.',
    'T7_biner': 'Hitung kolom bertipe integer yang nilai uniknya persis {0, 1}.',
    'T12_bulan': 'Setelah konversi yang benar, pakai .dt.month.nunique(). Data contoh ini hanya mencakup beberapa bulan.',
    'T12_rusak': "Bandingkan hasil konversi benar dengan pd.to_datetime(..., format='mixed', dayfirst=True) memakai !=, lalu jumlahkan.",
    'T8_rarity': 'Hitung baris dengan ~df.rarity.isin([4, 5]).',
    'T8_rate': 'Gabungkan kondisi di luar 0-1 untuk duplicate_rate dan c6_rate dengan | dan tanda kurung.',
    'T10_bisnis': 'Bandingkan pulled_count dengan penjumlahan constellation_0_pull sampai constellation_7_pull (sum axis=1), hitung yang tidak sama.',
    'T10_selisih': 'Ambil nilai mutlak selisih pulled_count dan jumlah C0-C7, lalu .max().',
    'T10_turunan': 'Hitung baris dengan selisih mutlak duplicate_rate dan (1 - C0/pulled_count) lebih dari 1e-6.'
}

def _h(x):
    return hashlib.sha256(str(x).strip().lower().encode()).hexdigest()[:10]

def cek(tugas, jawaban):
    """Periksa jawaban Anda tanpa menampilkan kuncinya."""
    if tugas not in _K:
        print('Nama tugas tidak dikenal:', tugas); return
    if _h(jawaban) == _K[tugas]:
        print(f'BENAR   {tugas} = {jawaban}')
    else:
        print(f'BELUM   {tugas} = {jawaban}')
        print('        Petunjuk:', _P.get(tugas, '-'))

def skor():
    """Tidak melacak apa pun; sekadar pengingat daftar tugas."""
    print('Daftar tugas:', ', '.join(sorted(_K)))

print('Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)')

Alat pemeriksa siap. Contoh pemakaian: cek("T1_baris", 1000)


---
## Bagian 1 — Lima dimensi kualitas data

Sebelum menyentuh kode, kenali apa saja yang sedang kita cari. Kualitas data biasanya dipecah menjadi lima dimensi.

| Dimensi | Pertanyaannya | Contoh masalah |
|---|---|---|
| **Kelengkapan** | Apakah ada nilai yang hilang? | Kolom kosong, atau placeholder seperti `-999` dan `N/A` |
| **Keunikan** | Apakah ada yang tercatat dua kali? | Baris duplikat persis maupun nyaris sama |
| **Konsistensi** | Apakah hal yang sama ditulis dengan cara yang sama? | `Liyue`, `liyue`, dan ` Liyue ` sebagai tiga wilayah berbeda; atau salah eja seperti `Extra-Terresterial` |
| **Validitas** | Apakah nilainya mungkin terjadi? | Rarity 7, *rate* bernilai 1,8 padahal harus 0–1 |
| **Akurasi** | Apakah nilainya benar? | `pulled_count` tidak sama dengan jumlah pull per konstelasi |

Urutan pemeriksaannya mengikuti urutan tabel ini, karena masalah di baris atas sering menyebabkan
salah tafsir pada baris bawah. Misalnya, menghitung rata-rata tanpa menyadari adanya placeholder
akan menghasilkan angka yang masuk akal secara tampilan tetapi salah total.

### Satu prinsip yang menentukan segalanya

> **Data yang kelihatan rapi belum tentu benar, dan pandas tidak akan memberi tahu Anda.**

`df.head()` akan tampil cantik untuk data seburuk apa pun. Satu-satunya cara mengetahui kondisi data
adalah memeriksanya satu dimensi demi satu dimensi, dengan sengaja.

Prinsip pasangannya: **hasil "tidak ditemukan masalah" harus dibuktikan, bukan diasumsikan.**
Dataset berlabel `clean` pun tetap harus diperiksa.

---
## Bagian 2 — Potret awal

Tujuh perintah ini adalah hal pertama yang dijalankan pada dataset apa pun, selamanya.

In [ ]:
# CONTOH 2.1 — tampilan permukaan
df.head(5)

,name,rarity,weapon,element,3D-model,region,roles,constellation_0_pull,constellation_1_pull,constellation_2_pull,constellation_3_pull,constellation_4_pull,constellation_5_pull,constellation_6_pull,constellation_7_pull,asc_stat_bonus_category,duplicate_rate,c6_rate,avg_copies_per_player,num_banners,is_standard_banner,pulled_count,kelompok_umur_rilis,gender
0,c6f281db386f,4,Sword,Anemo,Medium Female,Fontaine,"Off-Field,DPS",301409,240308,198054,149152,100295,81006,48706,219453,Anemo DMG Bonus,0.774796,0.036392,4.440421,3,1,1338383,2-3 Tahun,Female
1,b5813abc687d,5,Catalyst,Electro,Medium Female,Natlan,"On-Field,DPS",69219,11980,6303,1140,575,390,3353,90,CRIT Rate,0.256102,0.036035,1.344270,2,0,93049,<1 Tahun,Female
2,70526d434ba6,4,Claymore,Cryo,Medium Male,Liyue,"Off-Field,DPS,Support",1330794,971436,690870,433896,259920,158592,98847,332967,ATK%,0.688872,0.023110,3.214113,10,1,4277322,4+ Tahun,Male
3,14fdb506f4cb,5,Bow,Dendro,Medium Male,Sumeru,"On-Field,DPS",681601,30666,3741,588,200,108,959,90,Dendro DMG Bonus,0.050633,0.001336,1.053333,2,1,717953,3-4 Tahun,Male
4,dc6824714cde,5,Sword,Anemo,Tall Female,Mondstadt,"Off-Field,Survivability",1291930,44118,4584,572,65,36,42,66,Healing Bonus,0.036889,0.000031,1.038302,1,1,1341413,4+ Tahun,Female


In [ ]:
# CONTOH 2.2 — ukuran dan tipe
print('Ukuran:', df.shape)
print()
df.info()

Ukuran: (103, 24)

<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   name                     103 non-null    str    
 1   rarity                   103 non-null    int64  
 2   weapon                   103 non-null    str    
 3   element                  103 non-null    str    
 4   3D-model                 103 non-null    str    
 5   region                   103 non-null    str    
 6   roles                    103 non-null    str    
 7   constellation_0_pull     103 non-null    int64  
 8   constellation_1_pull     103 non-null    int64  
 9   constellation_2_pull     103 non-null    int64  
 10  constellation_3_pull     103 non-null    int64  
 11  constellation_4_pull     103 non-null    int64  
 12  constellation_5_pull     103 non-null    int64  
 13  constellation_6_pull     103 non-null    int64  
 14  constellation_7_pu

> **Amati kolom `Dtype` di atas.** Hal yang perlu dicatat:
>
> - Tidak ada kolom tanggal sama sekali. Informasi usia rilis hanya tersisa sebagai kelompok teks `kelompok_umur_rilis`.
> - Sembilan kolom bertipe teks. Wajar untuk nama dan kategori, tetapi `roles` dan `3D-model` perlu dicek isinya.
> - `is_standard_banner` bertipe `int64`, padahal isinya hanya 0 dan 1 (biner).
> - Kolom `Non-Null Count` penuh (103) di semua kolom. Itu **belum** membuktikan tidak ada kekosongan; placeholder tidak terhitung (Bagian 3).

In [ ]:
# CONTOH 2.3 — ringkasan numerik
df.describe().round(3)

,rarity,constellation_0_pull,constellation_1_pull,constellation_2_pull,constellation_3_pull,constellation_4_pull,constellation_5_pull,constellation_6_pull,constellation_7_pull,duplicate_rate,c6_rate,avg_copies_per_player,num_banners,is_standard_banner,pulled_count
count,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000
mean,4.563,533782.466,294590.272,221787.524,147776.893,96946.262,64379.650,50187.214,152643.010,0.442,0.034,2.543,4.835,0.515,1562093.165
std,0.498,492771.384,418771.281,325604.458,223934.698,147372.261,96980.602,65212.465,237282.022,0.283,0.021,1.612,3.487,0.502,1921421.954
min,4.000,15318.000,866.000,252.000,68.000,35.000,12.000,21.000,8.000,0.037,0.000,1.038,1.000,0.000,16854.000
25%,4.000,159055.000,22159.000,7525.500,1854.000,772.500,573.000,4970.000,154.500,0.193,0.024,1.239,2.000,0.000,224798.000
50%,5.000,318448.000,69352.000,30198.000,9124.000,4105.000,3780.000,16373.000,587.000,0.338,0.033,1.511,4.000,1.000,652085.000
75%,5.000,788685.500,446541.000,325993.500,229594.000,157910.000,109344.000,84416.500,235137.500,0.744,0.042,3.913,6.000,1.000,2261590.500
max,5.000,1775083.000,1581770.000,1213152.000,839452.000,562730.000,382812.000,261968.000,1042197.000,0.875,0.138,7.970,16.000,1.000,7659164.000


> **Lihat baris `min` dan `max`.** Tidak ada angka negatif, `rarity` hanya 4 dan 5, dan semua *rate* berada di antara 0 dan 1.
> Perhatikan juga selisih besar antara `mean` dan `50%` pada kolom *pull* (condong ke kanan: sedikit karakter sangat populer).
> Itu bukan cacat kualitas, tetapi karakter distribusinya; salah menafsirkannya sebagai cacat juga sebuah kesalahan.

In [ ]:
# CONTOH 2.4 — ringkasan kolom teks
df.describe(include=['object', 'string']).T

,count,unique,top,freq
name,103,103,c6f281db386f,1
weapon,103,5,Catalyst,24
element,103,7,Electro,17
3D-model,103,5,Medium Female,36
region,103,9,Liyue,21
roles,103,12,"On-Field,DPS",37
asc_stat_bonus_category,103,16,CRIT DMG,18
kelompok_umur_rilis,103,5,4+ Tahun,39
gender,103,2,Female,68


### TUGAS 1 — ukuran dataset

In [ ]:
# TUGAS 1
t1_baris = df.shape[0]   # jumlah baris
t1_kolom = df.shape[1]   # jumlah kolom

cek('T1_baris', t1_baris)
cek('T1_kolom', t1_kolom)

BENAR   T1_baris = 103
BENAR   T1_kolom = 24


---
## Bagian 3 — Kelengkapan: nilai kosong dan penyamarannya

Ada dua macam nilai kosong, dan hanya satu yang terlihat oleh `isna()`.

| Jenis | Contoh | Terdeteksi `isna()`? |
|---|---|---|
| Kosong sejati | `NaN`, sel kosong di CSV | Ya |
| Placeholder angka | `-999`, `0` yang berarti tidak diisi, `9999` | **Tidak** |
| Placeholder teks | `N/A`, `-`, `null`, `tidak diketahui`, `?` | **Tidak** |

Placeholder berbahaya justru karena ia terlihat seperti data sungguhan.

In [ ]:
# CONTOH 3.1 — kosong sejati
kosong = df.isna().sum()
print('Total sel kosong:', int(kosong.sum()))
kosong[kosong > 0].to_frame('jumlah_kosong').assign(
    persen=lambda d: (d.jumlah_kosong / len(df) * 100).round(1))

Total sel kosong: 0


,jumlah_kosong,persen


In [ ]:
# CONTOH 3.2 — berburu placeholder teks
CURIGA = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']

ada = False
for kol in df.select_dtypes(include=['object', 'string']).columns:
    nilai = df[kol].astype(str).str.strip().str.lower()
    ketemu = nilai[nilai.isin(CURIGA)].value_counts()
    if len(ketemu):
        ada = True
        print(f'{kol}:')
        print(ketemu.to_string(), '\n')
if not ada:
    print('Tidak ada placeholder teks pada kolom mana pun.')

Tidak ada placeholder teks pada kolom mana pun.


In [ ]:
# CONTOH 3.3 — berburu placeholder angka
# Caranya: lihat nilai ekstrem pada tiap kolom numerik
for kol in df.select_dtypes(include='number').columns:
    s = df[kol]
    print(f'{kol:24s} min={s.min():14,.4f}  max={s.max():16,.2f}  '
          f'terkecil unik: {sorted(s.dropna().unique())[:3]}')

rarity                   min=        4.0000  max=            5.00  terkecil unik: [np.int64(4), np.int64(5)]
constellation_0_pull     min=   15,318.0000  max=    1,775,083.00  terkecil unik: [np.int64(15318), np.int64(30630), np.int64(33078)]
constellation_1_pull     min=      866.0000  max=    1,581,770.00  terkecil unik: [np.int64(866), np.int64(2962), np.int64(3670)]
constellation_2_pull     min=      252.0000  max=    1,213,152.00  terkecil unik: [np.int64(252), np.int64(1560), np.int64(1680)]
constellation_3_pull     min=       68.0000  max=      839,452.00  terkecil unik: [np.int64(68), np.int64(108), np.int64(224)]
constellation_4_pull     min=       35.0000  max=      562,730.00  terkecil unik: [np.int64(35), np.int64(65), np.int64(70)]
constellation_5_pull     min=       12.0000  max=      382,812.00  terkecil unik: [np.int64(12), np.int64(18), np.int64(24)]
constellation_6_pull     min=       21.0000  max=      261,968.00  terkecil unik: [np.int64(21), np.int64(28), np.int64(

> **Pola yang perlu dihafal.** Placeholder angka hampir selalu berupa bilangan yang mencolok:
> `-1`, `-999`, `0`, `9999`, `99999`, atau tahun `1900`. Pada dataset ini tidak ada yang cocok dengan pola itu:
> nilai terkecil tiap kolom wajar (jumlah *pull* minimum puluhan sampai belasan ribu, bukan `-999`).
>
> Satu-satunya kolom yang berisi `0` adalah `is_standard_banner`, dan itu wajar karena memang kolom biner.

### TUGAS 2 — kelengkapan

In [ ]:
# TUGAS 2
df = pd.read_csv(FILE)

t2_kosong = df.isna().sum().sum()        # TOTAL sel kosong di seluruh dataframe
t2_kolom  = (df.isna().sum() > 0).sum()  # BERAPA KOLOM yang punya setidaknya satu nilai kosong

cek('T2_kosong', t2_kosong)
cek('T2_kolom', t2_kolom)

BENAR   T2_kosong = 0
BENAR   T2_kolom = 0


### TUGAS 3 — placeholder angka

In [ ]:
# TUGAS 3
PLACEHOLDER = [-999, -1, 9999, 99999, 999999]
num = df.select_dtypes(include='number')

t3_placeholder = num.isin(PLACEHOLDER).sum().sum()                                   # sel yang bernilai placeholder umum
t3_nol = (df.drop(columns='is_standard_banner').select_dtypes(include='number') == 0).sum().sum()  # nol pada kolom non-biner

cek('T3_placeholder', t3_placeholder)
cek('T3_nol', t3_nol)

BENAR   T3_placeholder = 0
BENAR   T3_nol = 0


---
## Bagian 4 — Mengapa data itu kosong: MCAR, MAR, MNAR

Menemukan nilai kosong baru separuh pekerjaan. Separuh lainnya adalah menebak **mengapa** kosong,
karena alasannya menentukan apakah Anda boleh menghapus baris, boleh mengisi, atau tidak boleh keduanya.

| Istilah | Artinya | Contoh | Akibatnya |
|---|---|---|---|
| **MCAR** | Kosong betul-betul acak | Kertas struk sobek | Menghapus baris aman, hanya kehilangan ukuran sampel |
| **MAR** | Kosong bergantung pada kolom lain **yang terlihat** | Kolom berat kosong untuk produk non-fisik | Bisa ditangani; isi per kelompok, atau akui bahwa kolom itu memang tidak berlaku |
| **MNAR** | Kosong bergantung pada **nilainya sendiri** yang tidak terlihat | Saldo besar sengaja tidak ditampilkan | Paling berbahaya; mengisi rata-rata akan menyesatkan |

> **Hal terpenting di bagian ini:** MAR dapat dibuktikan dari data. MCAR dan MNAR **tidak bisa dibedakan**
> hanya dengan melihat data, karena nilai yang hilang tidak dapat diperiksa. Membedakannya memerlukan
> pengetahuan tentang bagaimana data itu dikumpulkan.

**Dataset ini tidak punya nilai kosong**, jadi tidak ada pola untuk dianalisis. Agar tetap bisa berlatih, kita
**mensimulasikan** tiga jenis kekosongan pada *salinan* data. Data asli `df` tidak diubah. Aturan penghilangannya kita ketahui,
sehingga kita bisa melihat apa yang tampak (dan tidak tampak) oleh alat diagnosis.

In [ ]:
# CONTOH 4.1 — alat bantu: apakah kekosongan terkait kolom lain?
def pola_kosong(data, kolom_kosong, kolom_pembanding):
    """Persentase kekosongan kolom_kosong untuk tiap nilai kolom_pembanding."""
    return (data.groupby(kolom_pembanding)[kolom_kosong]
                .apply(lambda s: round(s.isna().mean() * 100, 1))
                .sort_values(ascending=False)
                .to_frame('persen_kosong'))

# Pada data asli: semuanya 0, karena memang tidak ada yang kosong
pola_kosong(df, 'c6_rate', 'rarity')

,persen_kosong
rarity,
4,0.0
5,0.0


In [ ]:
# CONTOH 4.2 — simulasi MCAR (salinan df; data asli utuh)
rng = np.random.default_rng(42)

df_mcar = df.copy()
df_mcar.loc[rng.random(len(df)) < 0.15, 'c6_rate'] = np.nan   # 15% dihapus ACAK

print('Total kosong c6_rate:', int(df_mcar['c6_rate'].isna().sum()))
print(pola_kosong(df_mcar, 'c6_rate', 'rarity'))

Total kosong c6_rate: 15
        persen_kosong
rarity               
5                17.2
4                11.1


> **Tampak MCAR.** Persentase kekosongan di kedua kelompok `rarity` berbeda tipis (selisih seperti ini wajar pada penghapusan acak dengan sampel kecil); tidak ada kolom teramati yang
> menjelaskannya. Menghapus baris atau mengabaikan kolom relatif aman (kita hanya kehilangan ukuran sampel).

In [ ]:
# CONTOH 4.3 — simulasi MAR
df_mar = df.copy()
df_mar.loc[df_mar['rarity'] == 4, 'c6_rate'] = np.nan   # kosong HANYA untuk bintang 4

print(pola_kosong(df_mar, 'c6_rate', 'rarity'))

        persen_kosong
rarity               
4               100.0
5                 0.0


> **Ini MAR, dan buktinya jelas.** `c6_rate` kosong 100% untuk `rarity` 4 dan 0% untuk `rarity` 5. Kekosongannya bergantung
> pada `rarity`, kolom yang bisa kita lihat. Tindakannya: akui bahwa kolom itu tidak tersedia untuk kelompok tersebut,
> bukan mengisinya dengan rata-rata seluruh data.

In [ ]:
# CONTOH 4.4 — simulasi MNAR: yang hilang justru nilai yang besar
batas = df['pulled_count'].quantile(0.80)

df_mnar = df.copy()
df_mnar.loc[df_mnar['pulled_count'] >= batas, 'pulled_count'] = np.nan   # 20% teratas disembunyikan

print('Kosong pulled_count:', int(df_mnar['pulled_count'].isna().sum()))
for kol in ['rarity', 'weapon', 'element']:
    print(f'--- pulled_count kosong menurut {kol} ---')
    print(pola_kosong(df_mnar, 'pulled_count', kol).head(3).to_string(), '\n')

print('Rata-rata ASLI    :', f"{df['pulled_count'].mean():,.0f}")
print('Rata-rata TERLIHAT:', f"{df_mnar['pulled_count'].mean():,.0f}")

Kosong pulled_count: 21
--- pulled_count kosong menurut rarity ---
        persen_kosong
rarity               
4                46.7
5                 0.0 

--- pulled_count kosong menurut weapon ---
          persen_kosong
weapon                 
Claymore           33.3
Bow                22.2
Catalyst           20.8 

--- pulled_count kosong menurut element ---
         persen_kosong
element               
Pyro              31.2
Geo               27.3
Electro           23.5 

Rata-rata ASLI    : 1,562,093
Rata-rata TERLIHAT: 693,439


> **Baca pelan-pelan, ini inti pertemuan hari ini.**
>
> Perhatikan jebakannya. Kekosongan tampak berkaitan dengan `rarity` (bintang 4 hilang sekitar 47%, bintang 5 tidak sama sekali),
> sehingga Anda akan tergoda menyimpulkan **MAR** dan mengisi per kelompok `rarity`. Padahal mekanisme sebenarnya adalah **nilainya sendiri**:
> yang hilang selalu yang terbesar, dan kebetulan bintang 4 punya *pull* terbesar. Rata-rata terlihat hanya sekitar 44% dari rata-rata aslinya.
>
> Dalam data nyata kita **tidak punya** rata-rata asli sebagai pembanding. Tidak ada uji statistik yang bisa memastikan
> MNAR. Yang bisa memastikannya hanya satu: **bertanya kepada pemilik data**.

### TUGAS 4 — membuktikan MAR dan mengukur bias MNAR

In [ ]:
# TUGAS 4
# a) Pada df_mar: berapa persen baris rarity 4 yang c6_rate-nya kosong? (1 desimal)
t4_mar = round(df_mar.loc[df_mar['rarity'] == 4, 'c6_rate'].isna().mean() * 100, 1)

# b) Rata-rata pulled_count TERLIHAT (df_mnar) sebagai persen dari rata-rata ASLI (df). (1 desimal)
t4_bias = round(df_mnar['pulled_count'].mean() / df['pulled_count'].mean() * 100, 1)

cek('T4_mar', t4_mar)
cek('T4_bias', t4_bias)

BENAR   T4_mar = 100.0
BENAR   T4_bias = 44.4


**Pertanyaan renungan (tulis jawabannya di sini, tidak ada sel cek):**

Untuk dataset proyek tim Anda, sebutkan satu kolom yang Anda duga MNAR, dan jelaskan
mengapa dugaan itu tidak bisa dibuktikan dari datanya sendiri.

**Jawaban:** Kolom `pulled_count` (dan turunannya `c6_rate`, `duplicate_rate`, `avg_copies_per_player`).
Angka *pull* per karakter hampir pasti berasal dari data yang dikirim sukarela oleh pemain (misalnya lewat pelacak riwayat *wish*), bukan
dari seluruh populasi pemain. Pemain yang rutin mengunggah riwayat cenderung pemain yang lebih aktif atau lebih serius,
sehingga kelompok pemain "pasif" tidak terwakili; ini kondisi mirip MNAR pada tingkat pengumpulan data (siapa yang masuk ke data
bergantung pada perilaku yang sedang diukur). Data tidak punya `NaN`, jadi tidak terlihat sebagai kolom kosong. Dugaan ini tidak bisa dibuktikan dari datanya sendiri karena
pemain yang tidak terekam tidak meninggalkan jejak apa pun; yang bisa memastikan hanya penjelasan metode pengumpulan dari sumber data.

---
## Bagian 5 — Keunikan: duplikat persis dan duplikat nyaris

Duplikat persis mudah ditemukan. Yang merepotkan adalah duplikat nyaris: baris yang merujuk kejadian
yang sama tetapi berbeda sedikit, misalnya beda huruf besar-kecil, atau semua isinya sama kecuali satu kolom.

In [ ]:
# CONTOH 5.1 — duplikat persis
print('Baris yang seluruh kolomnya sama:', df.duplicated().sum())
df[df.duplicated(keep=False)].head(6)

Baris yang seluruh kolomnya sama: 0


,name,rarity,weapon,element,3D-model,region,roles,constellation_0_pull,constellation_1_pull,constellation_2_pull,constellation_3_pull,constellation_4_pull,constellation_5_pull,constellation_6_pull,constellation_7_pull,asc_stat_bonus_category,duplicate_rate,c6_rate,avg_copies_per_player,num_banners,is_standard_banner,pulled_count,kelompok_umur_rilis,gender


In [ ]:
# CONTOH 5.2 — duplikat pada kunci utama
# name seharusnya unik (satu baris satu karakter). Kalau tidak, ada yang salah.
dup_nama = df['name'].duplicated(keep=False)
print('Baris dengan name berulang :', dup_nama.sum())
print('name kosong               :', df['name'].isna().sum())
print('name berulang (huruf kecil):', df['name'].str.lower().duplicated(keep=False).sum())

Baris dengan name berulang : 0
name kosong               : 0
name berulang (huruf kecil): 0


In [ ]:
# CONTOH 5.3 — duplikat nyaris: semua kolom sama KECUALI kunci
fitur = [k for k in df.columns if k != 'name']
print('Duplikat seluruh kolom      :', df.duplicated(keep=False).sum())
print('Duplikat menurut kunci name :', df.duplicated(subset=['name'], keep=False).sum())
print('Duplikat selain kolom name  :', df.duplicated(subset=fitur, keep=False).sum())
print()
print('Selisih antara ketiganya adalah duplikat NYARIS, yang justru paling sering luput.')

Duplikat seluruh kolom      : 0
Duplikat menurut kunci name : 0
Duplikat selain kolom name  : 0

Selisih antara ketiganya adalah duplikat NYARIS, yang justru paling sering luput.


> **Hasilnya bersih.** Ketiga pemeriksaan menghasilkan nol. Karena `name` sudah berupa kode acak (anonim),
> duplikat tidak bisa lagi dilihat dari nama, itulah sebabnya pemeriksaan "semua kolom kecuali `name`" penting: dua baris
> dengan angka *pull* identik tetapi kode berbeda adalah karakter yang sama yang tercatat dua kali.

### TUGAS 5 — keunikan

In [ ]:
# TUGAS 5
t5_dup     = df.duplicated().sum()                                          # keep='first' (default)
t5_dupnama = df['name'].duplicated(keep=False).sum()                       # name berulang, kedua pasangan ikut
t5_nyaris  = df.drop(columns='name').duplicated(keep=False).sum()          # sama persis kecuali name

cek('T5_dup', t5_dup)
cek('T5_dupnama', t5_dupnama)
cek('T5_nyaris', t5_nyaris)

BENAR   T5_dup = 0
BENAR   T5_dupnama = 0
BENAR   T5_nyaris = 0


---
## Bagian 6 — Konsistensi: satu hal, banyak tulisan

Masalah paling umum pada data yang diketik manusia, dan paling sering tidak disadari karena
dampaknya baru muncul saat agregasi.

In [ ]:
# CONTOH 6.1 — berapa wilayah sebenarnya?
df['region'].value_counts(dropna=False)

region
Liyue                 21
Mondstadt             18
Inazuma               17
Sumeru                14
Fontaine              13
Natlan                11
Nod-Krai               6
Snezhnaya              2
Extra-Terresterial     1
Name: count, dtype: int64

> Ada sembilan wilayah. Dua di antaranya sangat jarang: `Snezhnaya` (2 baris) dan `Extra-Terresterial` (1 baris).
> Daftar ini mudah dibaca karena pendek, tetapi kebersihan sebenarnya harus diuji dengan empat penyebab klasik:
> **huruf besar-kecil**, **spasi di awal atau akhir**, **sinonim**, dan **salah ketik**.

In [ ]:
# CONTOH 6.2 — memisahkan penyebabnya
s = df['region'].astype(str)
print('Nilai unik apa adanya            :', s.nunique())
print('Setelah dibuang spasinya         :', s.str.strip().nunique())
print('Setelah diseragamkan hurufnya    :', s.str.strip().str.title().nunique())
print()
print('Nilai langka (<= 2 baris):')
print(df['region'].value_counts()[lambda v: v <= 2].to_string())

Nilai unik apa adanya            : 9
Setelah dibuang spasinya         : 9
Setelah diseragamkan hurufnya    : 9

Nilai langka (<= 2 baris):
region
Snezhnaya             2
Extra-Terresterial    1


> Normalisasi mekanis (spasi, huruf) **tidak mengubah apa pun**: jumlahnya tetap sembilan. Tetapi ada satu
> masalah yang tidak bisa ditemukan kode: `Extra-Terresterial` hampir pasti salah eja dari `Extra-Terrestrial`.
> Komputer tidak tahu itu salah ketik; manusia yang harus memutuskan, lalu **mencatat alasannya**.
> Inilah sebabnya pertemuan 5 mewajibkan *cleaning log*.
>
> Dan `Snezhnaya` / `Extra-Terresterial` hanya muncul 1–2 kali: valid sebagai wilayah, tetapi akan menghasilkan
> rata-rata yang tidak stabil jika dianalisis per wilayah.

In [ ]:
# CONTOH 6.3 — spasi tersembunyi pada kolom kunci
a = df['name'].astype(str)
print('name unik apa adanya   :', a.nunique())
print('name unik setelah strip:', a.str.strip().nunique())
print('Selisih                :', a.nunique() - a.str.strip().nunique(), 'karakter hantu')
print('Format kode 12 heksadesimal:', a.str.fullmatch(r'[0-9a-f]{12}').sum(), 'dari', len(a))

name unik apa adanya   : 103
name unik setelah strip: 103
Selisih                : 0 karakter hantu
Format kode 12 heksadesimal: 103 dari 103


> **Dampaknya bisa nyata.** Pada data lain, spasi di akhir teks menciptakan entitas hantu yang membuat
> hitungan unik membengkak. Di sini selisihnya nol dan semua kode berformat sama, jadi kolom kunci ini sehat.

In [ ]:
# CONTOH 6.4 — periksa seluruh kolom teks sekaligus
bersih = True
for kol in df.select_dtypes(include=['object', 'string']).columns:
    s = df[kol].astype(str)
    mentah = s.nunique()
    rapi   = s.str.strip().str.lower().nunique()
    if mentah != rapi:
        bersih = False
        print(f'{kol:24s} {mentah:4d} -> {rapi:4d}   (selisih {mentah - rapi})')
print('Semua kolom teks konsisten terhadap spasi dan huruf.' if bersih else '')

Semua kolom teks konsisten terhadap spasi dan huruf.


### TUGAS 6 — konsistensi kategori

In [ ]:
# TUGAS 6
t6_kat       = df['region'].nunique()                                              # region APA ADANYA
t6_katbersih = df['region'].astype(str).str.strip().str.title().nunique()          # setelah strip dan title
t6_jarang    = (df['region'].value_counts() <= 2).sum()                            # kategori region dengan <= 2 baris

cek('T6_kat', t6_kat)
cek('T6_katbersih', t6_katbersih)
cek('T6_jarang', t6_jarang)

BENAR   T6_kat = 9


BENAR   T6_katbersih = 9
BENAR   T6_jarang = 2


### TUGAS 9 — kunci yang sehat (pengganti "anggota hantu")

In [ ]:
# TUGAS 9
t9_nama  = df['name'].nunique()                                    # nunique name apa adanya
t9_strip = df['name'].astype(str).str.strip().nunique()            # setelah strip
t9_hex   = df['name'].astype(str).str.fullmatch(r'[0-9a-f]{12}').sum()   # baris berformat 12 heksadesimal

cek('T9_nama', t9_nama)
cek('T9_strip', t9_strip)
cek('T9_hex', t9_hex)

BENAR   T9_nama = 103


BENAR   T9_strip = 103
BENAR   T9_hex = 103


---
## Bagian 7 — Tipe data yang salah

Tipe data yang salah tidak selalu berupa angka yang terbaca sebagai teks. Pada dataset ini ada tiga bentuk yang lebih halus.

In [ ]:
# CONTOH 7.1 — kolom bernilai majemuk: roles
print(df['roles'].head(5).tolist())
print()
print('Kombinasi unik       :', df['roles'].nunique())
print('Peran individual     :', sorted(df['roles'].str.split(',').explode().unique()))

['Off-Field,DPS', 'On-Field,DPS', 'Off-Field,DPS,Support', 'On-Field,DPS', 'Off-Field,Survivability']

Kombinasi unik       : 12
Peran individual     : ['DPS', 'Off-Field', 'On-Field', 'Support', 'Survivability']


> Satu sel `roles` berisi beberapa nilai yang dipisah koma. Selama masih satu string, Anda tidak bisa menjawab
> "berapa karakter yang berperan Support?" dengan `==`; hasilnya hanya cocok dengan kombinasi persis.
> Ini pelanggaran prinsip *tidy data* (satu sel, satu nilai). Pencatatan masalahnya sekarang, perbaikannya nanti.

In [ ]:
# CONTOH 7.2 — kolom yang menggabungkan dua atribut, dan kolom biner bertipe integer
print(pd.crosstab(df['3D-model'], df['gender']))
print()
for kol in df.select_dtypes(include='integer').columns:
    u = sorted(df[kol].unique())
    if len(u) <= 2:
        print(f'{kol}: nilai unik {u}, dtype {df[kol].dtype}')

gender         Female  Male
3D-model                   
Medium Female      36     0
Medium Male         0    21
Short Female       11     0
Tall Female        21     0
Tall Male           0    14

rarity: nilai unik [np.int64(4), np.int64(5)], dtype int64


is_standard_banner: nilai unik [np.int64(0), np.int64(1)], dtype int64


> `3D-model` menggabungkan ukuran tubuh dan jenis kelamin (`Medium Female`), dan `gender` hanya mengulang separuhnya:
> keduanya selalu cocok (tidak ada kontradiksi), tetapi redundan. `is_standard_banner` bertipe `int64`, padahal secara makna
> adalah `bool`. Tidak ada galat apa pun, hanya risiko salah tafsir (misalnya merata-ratakan kolom biner).

### Bagian paling berbahaya di seluruh workbook ini: tanggal dengan format campur

Dataset Genshin ini tidak punya kolom tanggal, tetapi pelajaran tanggal terlalu penting untuk dilewati: di dunia nyata
Anda pasti akan bertemu kolom tanggal dengan dua format bercampur. Kita pakai **contoh kecil buatan** (bukan bagian dataset).

Cara yang paling sering dipakai orang untuk menangani format campur adalah `format='mixed'`
ditambah `dayfirst=True`, karena di Indonesia tanggal memang ditulis hari lebih dulu.

In [ ]:
# CONTOH 7.3 — data contoh buatan: dua format bercampur
t = pd.Series([
    '2025-08-01 07:09:00', '2025-08-09 10:48:00', '2025-09-15 11:46:00',
    '2025-10-02 13:18:00', '2025-10-29 11:19:00', '2025-11-05 09:00:00',
    '01/09/2025 08:30',    '12/10/2025 14:05',    '25/10/2025 16:20',   '03/11/2025 10:10',
])
iso = t.str.match(r'^\d{4}-')
print('Berformat yyyy-mm-dd :', int(iso.sum()))
print('Berformat dd/mm/yyyy :', int((~iso).sum()))

# cara yang tampak benar
cara_umum = pd.to_datetime(t, format='mixed', dayfirst=True, errors='coerce')

print()
print('Gagal dikonversi :', int(cara_umum.isna().sum()))
print('Rentang tanggal  :', cara_umum.min(), 'sampai', cara_umum.max())
print('Bulan yang muncul:', sorted(int(b) for b in cara_umum.dt.month.dropna().unique()))

Berformat yyyy-mm-dd : 6
Berformat dd/mm/yyyy : 4

Gagal dikonversi : 0
Rentang tanggal  : 2025-01-08 07:09:00 sampai 2025-11-03 10:10:00
Bulan yang muncul: [1, 2, 5, 9, 10, 11]


> **Nol galat. Tidak ada peringatan apa pun. Dan hasilnya salah.**
>
> Data ini sebenarnya berisi transaksi **Agustus sampai November 2025** (empat bulan). Hasil di atas
> menyebar ke bulan yang tidak seharusnya ada.
>
> Penyebabnya: `dayfirst=True` ikut diterapkan pada baris yang formatnya sudah `yyyy-mm-dd`.
> Baris `2025-08-01` dibaca sebagai tahun 2025, **hari** 08, **bulan** 01, sehingga menjadi 8 Januari.
> Karena 08 dan 01 keduanya angka yang sah untuk hari maupun bulan, pandas tidak punya alasan mengeluh.
>
> Inilah jenis kesalahan yang paling mahal dalam analisis data: **salah yang tidak menimbulkan galat.**

In [ ]:
# CONTOH 7.4 — cara yang benar: tangani tiap format secara terpisah
bagian_iso = pd.to_datetime(t[iso],  format='%Y-%m-%d %H:%M:%S', errors='coerce')
bagian_dmy = pd.to_datetime(t[~iso], format='%d/%m/%Y %H:%M',    errors='coerce')
benar = pd.concat([bagian_iso, bagian_dmy]).sort_index()

print('Gagal dikonversi :', int(benar.isna().sum()))
print('Rentang tanggal  :', benar.min(), 'sampai', benar.max())
print('Bulan yang muncul:', sorted(int(b) for b in benar.dt.month.dropna().unique()))
print()
print('Baris yang berbeda antara cara umum dan cara benar:', int((cara_umum != benar).sum()))

Gagal dikonversi : 0
Rentang tanggal  : 2025-08-01 07:09:00 sampai 2025-11-05 09:00:00
Bulan yang muncul: [8, 9, 10, 11]

Baris yang berbeda antara cara umum dan cara benar: 4


> **Pelajaran yang dibawa pulang:** setelah setiap konversi tanggal, jalankan satu uji akal sehat.
> Yang paling murah adalah memeriksa bulan atau tahun yang muncul:
>
> ```python
> print(sorted(kolom.dt.month.unique()))
> print(kolom.min(), kolom.max())
> ```
>
> Kalau rentangnya tidak sesuai dengan yang Anda ketahui tentang data itu, konversinya salah —
> berapa pun jumlah galat yang muncul.

### TUGAS 7 dan 12 — tipe data

In [ ]:
# TUGAS 7
t7_peran     = df['roles'].str.split(',').explode().nunique()          # peran individual
t7_kombinasi = df['roles'].nunique()                                    # kombinasi apa adanya
t7_biner     = sum(set(df[k].unique()) == {0, 1} for k in df.select_dtypes(include='integer').columns)  # kolom int yang biner

cek('T7_peran', t7_peran)
cek('T7_kombinasi', t7_kombinasi)
cek('T7_biner', t7_biner)

BENAR   T7_peran = 5
BENAR   T7_kombinasi = 12
BENAR   T7_biner = 1


In [ ]:
# TUGAS 12 (pada data contoh tanggal di CONTOH 7.3-7.4)
t12_bulan = benar.dt.month.nunique()    # bulan berbeda pada konversi yang BENAR
t12_rusak = (cara_umum != benar).sum()  # baris yang berbeda antara cara umum dan cara benar

cek('T12_bulan', t12_bulan)
cek('T12_rusak', t12_rusak)

BENAR   T12_bulan = 4
BENAR   T12_rusak = 4


---
## Bagian 8 — Validitas dan akurasi: aturan yang harus dipenuhi data

Dua pemeriksaan terakhir tidak bisa ditebak dari datanya saja. Keduanya berasal dari **pengetahuan
tentang hal yang diukur**, dan karena itu harus Anda tuliskan sendiri sebagai daftar aturan.

In [ ]:
# CONTOH 8.1 — aturan rentang
aturan = {
    'rarity harus 4 atau 5'         : ~df['rarity'].isin([4, 5]),
    'duplicate_rate harus 0-1'      : (df['duplicate_rate'] < 0) | (df['duplicate_rate'] > 1),
    'c6_rate harus 0-1'             : (df['c6_rate'] < 0) | (df['c6_rate'] > 1),
    'is_standard_banner harus 0/1'  : ~df['is_standard_banner'].isin([0, 1]),
    'num_banners harus >= 1'        : df['num_banners'] < 1,
    'avg_copies harus >= 1'         : df['avg_copies_per_player'] < 1,
    'jumlah pull harus > 0'         : (df.filter(like='_pull') <= 0).any(axis=1) | (df['pulled_count'] <= 0),
}
for nama, pelanggar in aturan.items():
    print(f'{nama:32s} dilanggar {int(pelanggar.sum()):4d} baris')

rarity harus 4 atau 5            dilanggar    0 baris
duplicate_rate harus 0-1         dilanggar    0 baris
c6_rate harus 0-1                dilanggar    0 baris
is_standard_banner harus 0/1     dilanggar    0 baris
num_banners harus >= 1           dilanggar    0 baris
avg_copies harus >= 1            dilanggar    0 baris
jumlah pull harus > 0            dilanggar    0 baris


In [ ]:
# CONTOH 8.2 — aturan hubungan antarkolom
KOL_C = [f'constellation_{i}_pull' for i in range(8)]
total_C = df[KOL_C].sum(axis=1)
selisih = (df['pulled_count'] - total_C)

tidak_cocok = selisih != 0
print('Baris yang tidak cocok         :', int(tidak_cocok.sum()), 'dari', len(df))
print('Selisih terbesar (mutlak)      :', int(selisih.abs().max()))
print()
df.loc[tidak_cocok, ['pulled_count']].assign(jumlah_C0_C7=total_C[tidak_cocok], selisih=selisih[tidak_cocok]).head(9)

Baris yang tidak cocok         : 9 dari 103
Selisih terbesar (mutlak)      : 3



,pulled_count,jumlah_C0_C7,selisih
1,93049,93050,-1
8,629343,629344,-1
9,1490123,1490124,-1
12,2798153,2798155,-2
15,1838324,1838325,-1
22,6183960,6183962,-2
28,3761404,3761405,-1
40,2092017,2092020,-3
58,669225,669226,-1


> **Perhatikan sifatnya.** Sembilan baris tidak cocok, tetapi selisihnya hanya 1 sampai 3 pada angka jutaan: pola khas
> pembulatan atau sinkronisasi data yang tidak serempak. Secara bisnis kecil, tetapi tetap **catat**: jawabannya "kemungkinan
> pembulatan", bukan "pasti pembulatan". Yang bisa memastikan hanya pemilik data.

In [ ]:
# CONTOH 8.3 — kolom turunan: apakah rumus antarkolom konsisten?
turun_dup = (df['duplicate_rate'] - (1 - df['constellation_0_pull'] / df['pulled_count'])).abs() > 1e-6
turun_avg = (df['avg_copies_per_player'] - df['pulled_count'] / df['constellation_0_pull']).abs() > 1e-6
turun_c6  = (df['c6_rate'] - df['constellation_6_pull'] / df['pulled_count']).abs() > 1e-6

print('duplicate_rate != 1 - C0/pulled :', int(turun_dup.sum()))
print('avg_copies != pulled/C0         :', int(turun_avg.sum()))
print('c6_rate != C6/pulled            :', int(turun_c6.sum()))

duplicate_rate != 1 - C0/pulled : 0
avg_copies != pulled/C0         : 0
c6_rate != C6/pulled            : 0


> Ketiga kolom turunan konsisten dengan kolom asalnya, jadi ini hasil **lolos**. Kalau salah satu tidak lolos, kita akan
> tahu bahwa rumusnya berbeda dari dugaan atau ada baris yang rusak.

In [ ]:
# CONTOH 8.4 — pencilan: diagnosis, bukan penghapusan
def iqr_pencilan(s):
    q1, q3 = s.quantile([.25, .75])
    i = q3 - q1
    return int(((s < q1 - 1.5 * i) | (s > q3 + 1.5 * i)).sum())

for kol in ['pulled_count', 'c6_rate', 'num_banners', 'avg_copies_per_player', 'duplicate_rate']:
    print(f'{kol:24s} pencilan IQR: {iqr_pencilan(df[kol])}')

pulled_count             pencilan IQR: 9
c6_rate                  pencilan IQR: 5
num_banners              pencilan IQR: 4
avg_copies_per_player    pencilan IQR: 1
duplicate_rate           pencilan IQR: 0


> Pencilan **bukan** otomatis salah. `pulled_count` besar mungkin memang karakter yang sangat populer. Diagnosis hari ini
> hanya mencatat; keputusan memakai atau membuang butuh alasan bisnis (pertemuan 5).

### TUGAS 8 dan 10 — validitas dan aturan bisnis

In [ ]:
# TUGAS 8
t8_rarity = (~df['rarity'].isin([4, 5])).sum()                                           # rarity bukan 4/5
t8_rate   = (((df['duplicate_rate'] < 0) | (df['duplicate_rate'] > 1)) |
             ((df['c6_rate'] < 0) | (df['c6_rate'] > 1))).sum()                           # rate di luar 0-1

cek('T8_rarity', t8_rarity)
cek('T8_rate', t8_rate)

BENAR   T8_rarity = 0
BENAR   T8_rate = 0


In [ ]:
# TUGAS 10
KOL_C = [f'constellation_{i}_pull' for i in range(8)]
total_C = df[KOL_C].sum(axis=1)

t10_bisnis  = (df['pulled_count'] != total_C).sum()                                       # baris pulled_count != jumlah C0-C7
t10_selisih = (df['pulled_count'] - total_C).abs().max()                                  # selisih mutlak terbesar
t10_turunan = ((df['duplicate_rate'] - (1 - df['constellation_0_pull'] / df['pulled_count'])).abs() > 1e-6).sum()

cek('T10_bisnis', t10_bisnis)
cek('T10_selisih', t10_selisih)
cek('T10_turunan', t10_turunan)

BENAR   T10_bisnis = 9
BENAR   T10_selisih = 3
BENAR   T10_turunan = 0


---
## Bagian 9 — Merangkai temuan menjadi profil kualitas data

Pemeriksaan yang berserakan di banyak sel tidak berguna bagi siapa pun. Rangkum menjadi satu tabel
yang bisa dibaca orang lain dalam satu menit. Tabel inilah inti **M2**.

In [ ]:
# CONTOH 9.1 — kerangka profil kualitas data
temuan = []

def catat(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan.append({
        'dimensi'   : dimensi,
        'kolom'     : kolom,
        'masalah'   : masalah,
        'jumlah'    : int(jumlah),
        'persen'    : round(jumlah / len(df) * 100, 2),
        'keparahan' : keparahan,          # Tinggi / Sedang / Rendah
        'rencana'   : rencana,            # diisi lengkap pada pertemuan 5
    })

catat('Akurasi', 'pulled_count', 'Tidak sama dengan jumlah C0-C7 (selisih 1-3, kemungkinan pembulatan)',
      (df['pulled_count'] != total_C).sum(), 'Rendah', 'Tanyakan cara pulled_count dihitung; jangan diubah dulu')
catat('Konsistensi', 'region', 'Salah eja: Extra-Terresterial (seharusnya Extra-Terrestrial)',
      (df['region'] == 'Extra-Terresterial').sum(), 'Sedang', 'Koreksi ejaan, catat di cleaning log')
catat('Konsistensi', 'region', 'Kategori sangat jarang (<= 2 baris): Snezhnaya, Extra-Terresterial',
      df['region'].isin(df['region'].value_counts()[lambda v: v <= 2].index).sum(),
      'Rendah', 'Putuskan: dipertahankan, atau digabung ke Lainnya untuk analisis per wilayah')
catat('Validitas', 'roles', 'Beberapa nilai dalam satu sel (dipisah koma)',
      len(df), 'Sedang', 'Pecah menjadi kolom biner per peran')
catat('Validitas', '3D-model / gender', 'Atribut gabungan dan redundan',
      len(df), 'Rendah', 'Pisahkan ukuran tubuh; pertahankan satu sumber gender')
catat('Kelengkapan', 'seluruh kolom', 'Tidak ditemukan NaN maupun placeholder (lolos)',
      0, 'Rendah', 'Tidak ada tindakan')

profil = pd.DataFrame(temuan)
urut = {'Tinggi': 0, 'Sedang': 1, 'Rendah': 2}
profil = profil.sort_values(['keparahan', 'jumlah'], key=lambda c: c.map(urut) if c.name == 'keparahan' else -c)
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
3,Validitas,roles,Beberapa nilai dalam satu sel (dipisah koma),103,100.00,Sedang,Pecah menjadi kolom biner per peran
1,Konsistensi,region,Salah eja: Extra-Terresterial (seharusnya Extr...,1,0.97,Sedang,"Koreksi ejaan, catat di cleaning log"
4,Validitas,3D-model / gender,Atribut gabungan dan redundan,103,100.00,Rendah,Pisahkan ukuran tubuh; pertahankan satu sumber...
0,Akurasi,pulled_count,"Tidak sama dengan jumlah C0-C7 (selisih 1-3, k...",9,8.74,Rendah,Tanyakan cara pulled_count dihitung; jangan di...
2,Konsistensi,region,Kategori sangat jarang (<= 2 baris): Snezhnaya...,3,2.91,Rendah,"Putuskan: dipertahankan, atau digabung ke Lain..."
5,Kelengkapan,seluruh kolom,Tidak ditemukan NaN maupun placeholder (lolos),0,0.00,Rendah,Tidak ada tindakan


### TUGAS 11 — lengkapi profilnya

Tambahkan **minimal empat temuan lagi** dengan memanggil `catat(...)`, lalu tampilkan ulang tabelnya.
Tidak ada sel `cek` untuk tugas ini — penilaiannya memakai Rubrik M2.

In [ ]:
# TUGAS 11
catat('Validitas', 'is_standard_banner', 'Kolom biner bertipe int64, bukan boolean',
      len(df), 'Rendah', 'Konversi ke bool atau category pada pertemuan 5')
catat('Konsistensi', 'constellation_6_pull / constellation_7_pull', 'C7 lebih besar daripada C6 pada banyak baris; makna kolom perlu dikonfirmasi',
      (df['constellation_7_pull'] > df['constellation_6_pull']).sum(), 'Sedang', 'Tanyakan apakah C7 = kumulatif "C7 ke atas"; jangan dijumlahkan sebelum jelas')
catat('Validitas', 'pulled_count', 'Pencilan menurut IQR (belum tentu salah)',
      iqr_pencilan(df['pulled_count']), 'Rendah', 'Pertahankan; analisis dengan skala log atau median')
catat('Validitas', 'c6_rate', 'Pencilan menurut IQR (belum tentu salah)',
      iqr_pencilan(df['c6_rate']), 'Rendah', 'Telusuri karakter terkait sebelum memutuskan')
catat('Kelengkapan', 'pulled_count', 'Dugaan bias pengumpulan (MNAR pada level sumber data)',
      0, 'Tinggi', 'Tanyakan metode pengumpulan ke pemilik data; tidak bisa dibuktikan dari data')
catat('Keunikan', 'seluruh kolom / name', 'Tidak ada duplikat persis maupun nyaris (lolos)',
      0, 'Rendah', 'Tidak ada tindakan')

profil = pd.DataFrame(temuan)
profil = profil.sort_values(['keparahan', 'jumlah'], key=lambda c: c.map(urut) if c.name == 'keparahan' else -c)
profil

,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
10,Kelengkapan,pulled_count,Dugaan bias pengumpulan (MNAR pada level sumbe...,0,0.00,Tinggi,Tanyakan metode pengumpulan ke pemilik data; t...
3,Validitas,roles,Beberapa nilai dalam satu sel (dipisah koma),103,100.00,Sedang,Pecah menjadi kolom biner per peran
7,Konsistensi,constellation_6_pull / constellation_7_pull,C7 lebih besar daripada C6 pada banyak baris; ...,50,48.54,Sedang,"Tanyakan apakah C7 = kumulatif ""C7 ke atas""; j..."
1,Konsistensi,region,Salah eja: Extra-Terresterial (seharusnya Extr...,1,0.97,Sedang,"Koreksi ejaan, catat di cleaning log"
4,Validitas,3D-model / gender,Atribut gabungan dan redundan,103,100.00,Rendah,Pisahkan ukuran tubuh; pertahankan satu sumber...
6,Validitas,is_standard_banner,"Kolom biner bertipe int64, bukan boolean",103,100.00,Rendah,Konversi ke bool atau category pada pertemuan 5
0,Akurasi,pulled_count,"Tidak sama dengan jumlah C0-C7 (selisih 1-3, k...",9,8.74,Rendah,Tanyakan cara pulled_count dihitung; jangan di...
8,Validitas,pulled_count,Pencilan menurut IQR (belum tentu salah),9,8.74,Rendah,Pertahankan; analisis dengan skala log atau me...
9,Validitas,c6_rate,Pencilan menurut IQR (belum tentu salah),5,4.85,Rendah,Telusuri karakter terkait sebelum memutuskan
2,Konsistensi,region,Kategori sangat jarang (<= 2 baris): Snezhnaya...,3,2.91,Rendah,"Putuskan: dipertahankan, atau digabung ke Lain..."


In [ ]:
# Simpan profil Anda
profil.to_csv('profil_kualitas_latihan.csv', index=False)
print('Tersimpan:', profil.shape)

Tersimpan: (12, 7)


---
## Bagian 10 — Terapkan pada data proyek tim

**Ini bagian yang dinilai.** Hasilnya menjadi bagian pertama dari **M2 — Data Quality Report**
yang dikumpulkan pada pertemuan 5.

Dataset proyek tim di sini adalah **`genshin_impact_clean.csv`**. Dibaca ke variabel terpisah `df_tim` supaya
`df` latihan tidak tertimpa.

In [ ]:
# 10.1 — muat dataset proyek tim
df_tim = pd.read_csv('genshin_impact_clean.csv')
print('Berhasil dimuat:', df_tim.shape)
df_tim.info()

Berhasil dimuat: (103, 24)


<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   name                     103 non-null    str    
 1   rarity                   103 non-null    int64  
 2   weapon                   103 non-null    str    
 3   element                  103 non-null    str    
 4   3D-model                 103 non-null    str    
 5   region                   103 non-null    str    
 6   roles                    103 non-null    str    
 7   constellation_0_pull     103 non-null    int64  
 8   constellation_1_pull     103 non-null    int64  
 9   constellation_2_pull     103 non-null    int64  
 10  constellation_3_pull     103 non-null    int64  
 11  constellation_4_pull     103 non-null    int64  
 12  constellation_5_pull     103 non-null    int64  
 13  constellation_6_pull     103 non-null    int64  
 14  constellation_7_pull     103 non-null

### 10.2 — sembilan pemeriksaan pada data tim
Urutannya mengikuti Bagian 2–8. Hasil tiap pemeriksaan dicatat di sel penutup 10.3.

In [ ]:
# 1. Potret awal
display(df_tim.head(5))
df_tim.describe().round(3)

,name,rarity,weapon,element,3D-model,region,roles,constellation_0_pull,constellation_1_pull,constellation_2_pull,constellation_3_pull,constellation_4_pull,constellation_5_pull,constellation_6_pull,constellation_7_pull,asc_stat_bonus_category,duplicate_rate,c6_rate,avg_copies_per_player,num_banners,is_standard_banner,pulled_count,kelompok_umur_rilis,gender
0,c6f281db386f,4,Sword,Anemo,Medium Female,Fontaine,"Off-Field,DPS",301409,240308,198054,149152,100295,81006,48706,219453,Anemo DMG Bonus,0.774796,0.036392,4.440421,3,1,1338383,2-3 Tahun,Female
1,b5813abc687d,5,Catalyst,Electro,Medium Female,Natlan,"On-Field,DPS",69219,11980,6303,1140,575,390,3353,90,CRIT Rate,0.256102,0.036035,1.344270,2,0,93049,<1 Tahun,Female
2,70526d434ba6,4,Claymore,Cryo,Medium Male,Liyue,"Off-Field,DPS,Support",1330794,971436,690870,433896,259920,158592,98847,332967,ATK%,0.688872,0.023110,3.214113,10,1,4277322,4+ Tahun,Male
3,14fdb506f4cb,5,Bow,Dendro,Medium Male,Sumeru,"On-Field,DPS",681601,30666,3741,588,200,108,959,90,Dendro DMG Bonus,0.050633,0.001336,1.053333,2,1,717953,3-4 Tahun,Male
4,dc6824714cde,5,Sword,Anemo,Tall Female,Mondstadt,"Off-Field,Survivability",1291930,44118,4584,572,65,36,42,66,Healing Bonus,0.036889,0.000031,1.038302,1,1,1341413,4+ Tahun,Female


,rarity,constellation_0_pull,constellation_1_pull,constellation_2_pull,constellation_3_pull,constellation_4_pull,constellation_5_pull,constellation_6_pull,constellation_7_pull,duplicate_rate,c6_rate,avg_copies_per_player,num_banners,is_standard_banner,pulled_count
count,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000,103.000
mean,4.563,533782.466,294590.272,221787.524,147776.893,96946.262,64379.650,50187.214,152643.010,0.442,0.034,2.543,4.835,0.515,1562093.165
std,0.498,492771.384,418771.281,325604.458,223934.698,147372.261,96980.602,65212.465,237282.022,0.283,0.021,1.612,3.487,0.502,1921421.954
min,4.000,15318.000,866.000,252.000,68.000,35.000,12.000,21.000,8.000,0.037,0.000,1.038,1.000,0.000,16854.000
25%,4.000,159055.000,22159.000,7525.500,1854.000,772.500,573.000,4970.000,154.500,0.193,0.024,1.239,2.000,0.000,224798.000
50%,5.000,318448.000,69352.000,30198.000,9124.000,4105.000,3780.000,16373.000,587.000,0.338,0.033,1.511,4.000,1.000,652085.000
75%,5.000,788685.500,446541.000,325993.500,229594.000,157910.000,109344.000,84416.500,235137.500,0.744,0.042,3.913,6.000,1.000,2261590.500
max,5.000,1775083.000,1581770.000,1213152.000,839452.000,562730.000,382812.000,261968.000,1042197.000,0.875,0.138,7.970,16.000,1.000,7659164.000


In [ ]:
# 2. Nilai kosong sejati
kosong_tim = df_tim.isna().sum()
print('Total sel kosong:', int(kosong_tim.sum()), '| kolom terdampak:', int((kosong_tim > 0).sum()))

Total sel kosong:

 0 | kolom terdampak: 0


In [ ]:
# 3. Placeholder teks dan angka
CURIGA_TIM = ['n/a', 'na', '-', '', 'null', 'none', 'tidak diketahui', 'unknown', '?', '#n/a']
teks = df_tim.select_dtypes(include=['object', 'string']).columns
jumlah_teks = sum(df_tim[k].astype(str).str.strip().str.lower().isin(CURIGA_TIM).sum() for k in teks)
num_tim = df_tim.select_dtypes(include='number')
jumlah_angka = num_tim.isin([-999, -1, 9999, 99999, 999999]).sum().sum()
print('Placeholder teks  :', int(jumlah_teks))
print('Placeholder angka :', int(jumlah_angka))

Placeholder teks  : 0
Placeholder angka : 0


### Nomor 4 — Pola kekosongan (MCAR / MAR / MNAR)

Tidak ada kolom dengan nilai kosong di atas 5% (bahkan tidak ada satu pun nilai kosong), sehingga tidak ada kolom yang
perlu diklasifikasikan MCAR atau MAR dari datanya. Namun kesimpulan **"tidak ada NaN"** tidak sama dengan **"data lengkap"**:

| Kolom | Penilaian | Alasan |
|---|---|---|
| Seluruh kolom (kekosongan teramati) | Tidak ada | `isna()` nol; placeholder teks dan angka juga nol |
| `pulled_count` dan turunannya | **Dicurigai MNAR pada level sumber data** | Data *pull* kemungkinan berasal dari pemain yang mau mengunggah riwayat, bukan seluruh pemain. Tidak bisa dibuktikan dari data; perlu bertanya ke pemilik data (lihat renungan Bagian 4). |
| `region` / kolom kategori | Tidak ada indikasi | Tidak ada nilai hilang maupun placeholder |

In [ ]:
# 5. Keunikan
print('Duplikat persis          :', df_tim.duplicated().sum())
print('name berulang            :', df_tim['name'].duplicated(keep=False).sum())
print('Duplikat selain kolom name:', df_tim.drop(columns='name').duplicated(keep=False).sum())

Duplikat persis          : 0
name berulang            : 0
Duplikat selain kolom name: 0


In [ ]:
# 6. Konsistensi kolom teks
for kol in ['weapon', 'element', 'region', 'kelompok_umur_rilis', 'gender']:
    print(f'--- {kol} ---')
    print(df_tim[kol].value_counts().to_string(), '\n')
for kol in teks:
    s = df_tim[kol].astype(str)
    if s.nunique() != s.str.strip().str.lower().nunique():
        print('Tidak konsisten (spasi/huruf):', kol)
print('Cek ejaan manual: region mengandung "Extra-Terresterial" ->', (df_tim['region'] == 'Extra-Terresterial').sum(), 'baris')

--- weapon ---
weapon
Catalyst    24
Sword       22
Polearm     21
Claymore    18
Bow         18 

--- element ---
element
Electro    17
Anemo      16
Cryo       16
Pyro       16
Hydro      15
Dendro     12
Geo        11 

--- region ---
region
Liyue                 21
Mondstadt             18
Inazuma               17
Sumeru                14
Fontaine              13
Natlan                11
Nod-Krai               6
Snezhnaya              2
Extra-Terresterial     1 

--- kelompok_umur_rilis ---


kelompok_umur_rilis
4+ Tahun     39
<1 Tahun     17
3-4 Tahun    17
2-3 Tahun    15
1-2 Tahun    15 

--- gender ---
gender
Female    68
Male      35 

Cek ejaan manual: region mengandung "Extra-Terresterial" -> 1 baris


In [ ]:
# 7. Tipe data yang salah
print(df_tim.dtypes.value_counts().to_string(), '\n')
print('roles bernilai majemuk  :', df_tim['roles'].str.contains(',').sum(), 'baris berisi koma')
print('is_standard_banner      :', sorted(df_tim['is_standard_banner'].unique()), '-> biner bertipe', df_tim['is_standard_banner'].dtype)
print('kelompok_umur_rilis     :', df_tim['kelompok_umur_rilis'].dtype, '(ordinal tetapi tersimpan sebagai teks biasa)')
print('Kolom tanggal           :', [k for k in df_tim.columns if 'date' in k.lower() or 'tanggal' in k.lower()] or 'tidak ada')

int64      12
str         9
float64     3 

roles bernilai majemuk  : 103 baris berisi koma
is_standard_banner      : [np.int64(0), np.int64(1)] -> biner bertipe int64
kelompok_umur_rilis     : str (ordinal tetapi tersimpan sebagai teks biasa)
Kolom tanggal           : tidak ada


### Nomor 8 & 9 — Aturan validitas rentang dan hubungan antarkolom

Aturan yang ditulis tim berdasarkan pengetahuan tentang data:

1. `rarity` harus 4 atau 5.
2. `duplicate_rate` harus 0–1 (proporsi).
3. `c6_rate` harus 0–1 (proporsi).
4. `is_standard_banner` harus 0 atau 1.
5. `num_banners` harus >= 1 dan seluruh jumlah *pull* harus > 0.
6. `avg_copies_per_player` harus >= 1 (pemain yang punya karakter minimal punya satu salinan).
7. Karakter bintang 4 selalu tampil di banner standar: jika `rarity` = 4 maka `is_standard_banner` harus 1.
8. `gender` harus konsisten dengan `3D-model`.
9. Hubungan: `pulled_count` = jumlah `constellation_0_pull` sampai `constellation_7_pull`.
10. Hubungan turunan: `duplicate_rate`, `avg_copies_per_player`, dan `c6_rate` dapat dihitung dari kolom pull.

In [ ]:
# 8. Aturan rentang dan aturan pengetahuan domain
kol_c = [f'constellation_{i}_pull' for i in range(8)]
aturan_tim = {
    '1 rarity 4 atau 5'                 : ~df_tim['rarity'].isin([4, 5]),
    '2 duplicate_rate 0-1'              : (df_tim['duplicate_rate'] < 0) | (df_tim['duplicate_rate'] > 1),
    '3 c6_rate 0-1'                     : (df_tim['c6_rate'] < 0) | (df_tim['c6_rate'] > 1),
    '4 is_standard_banner 0/1'          : ~df_tim['is_standard_banner'].isin([0, 1]),
    '5 num_banners >= 1, pull > 0'      : (df_tim['num_banners'] < 1) | (df_tim[kol_c + ['pulled_count']] <= 0).any(axis=1),
    '6 avg_copies >= 1'                 : df_tim['avg_copies_per_player'] < 1,
    '7 rarity 4 -> banner standar'      : (df_tim['rarity'] == 4) & (df_tim['is_standard_banner'] != 1),
    '8 gender sesuai 3D-model'          : df_tim['3D-model'].str.contains('Female') != (df_tim['gender'] == 'Female'),
}
for nama, pelanggar in aturan_tim.items():
    print(f'{nama:34s} dilanggar {int(pelanggar.sum()):4d} baris')

1 rarity 4 atau 5                  dilanggar    0 baris


2 duplicate_rate 0-1               dilanggar    0 baris
3 c6_rate 0-1                      dilanggar    0 baris
4 is_standard_banner 0/1           dilanggar    0 baris
5 num_banners >= 1, pull > 0       dilanggar    0 baris
6 avg_copies >= 1                  dilanggar    0 baris
7 rarity 4 -> banner standar       dilanggar    0 baris
8 gender sesuai 3D-model           dilanggar    0 baris


In [ ]:
# 9. Aturan hubungan antarkolom
total_C = df_tim[kol_c].sum(axis=1)
selisih = df_tim['pulled_count'] - total_C
tidak_sama = selisih != 0
print('9  pulled_count != jumlah C0-C7      :', int(tidak_sama.sum()), 'baris | selisih mutlak maks:', int(selisih.abs().max()))

dup_t = (df_tim['duplicate_rate'] - (1 - df_tim['constellation_0_pull'] / df_tim['pulled_count'])).abs() > 1e-6
avg_t = (df_tim['avg_copies_per_player'] - df_tim['pulled_count'] / df_tim['constellation_0_pull']).abs() > 1e-6
c6_t  = (df_tim['c6_rate'] - df_tim['constellation_6_pull'] / df_tim['pulled_count']).abs() > 1e-6
print('10 duplicate_rate turunan salah      :', int(dup_t.sum()))
print('10 avg_copies turunan salah          :', int(avg_t.sum()))
print('10 c6_rate turunan salah             :', int(c6_t.sum()))
print()
print('Catatan: C7 > C6 pada', int((df_tim['constellation_7_pull'] > df_tim['constellation_6_pull']).sum()), 'baris (makna C7 perlu dikonfirmasi).')

9  pulled_count != jumlah C0-C7      : 9 baris | selisih mutlak maks: 3
10 duplicate_rate turunan salah      : 0
10 avg_copies turunan salah          : 0
10 c6_rate turunan salah             : 0

Catatan: C7 > C6 pada 50 baris (makna C7 perlu dikonfirmasi).


In [ ]:
# 10.3 — susun profil kualitas data tim
os.makedirs('reports', exist_ok=True)
temuan_tim = []

def catat_tim(dimensi, kolom, masalah, jumlah, keparahan, rencana):
    temuan_tim.append({
        'dimensi': dimensi, 'kolom': kolom, 'masalah': masalah,
        'jumlah': int(jumlah), 'persen': round(jumlah / len(df_tim) * 100, 2),
        'keparahan': keparahan, 'rencana': rencana,
    })

catat_tim('Kelengkapan', 'pulled_count dan turunannya', 'Dugaan bias pengumpulan (MNAR di level sumber data); tidak terlihat sebagai NaN',
          0, 'Tinggi', 'Tanyakan metode pengumpulan ke pemilik data; tidak bisa dibuktikan dari data')
catat_tim('Akurasi', 'pulled_count', 'Tidak sama dengan jumlah C0-C7 (selisih 1-3, diduga pembulatan)',
          tidak_sama.sum(), 'Rendah', 'Konfirmasi cara hitung; jangan diubah sebelum jelas')
catat_tim('Konsistensi', 'region', 'Salah eja Extra-Terresterial',
          (df_tim['region'] == 'Extra-Terresterial').sum(), 'Sedang', 'Koreksi menjadi Extra-Terrestrial; catat di cleaning log')
catat_tim('Konsistensi', 'region', 'Kategori sangat jarang (<= 2 baris)',
          df_tim['region'].isin(df_tim['region'].value_counts()[lambda v: v <= 2].index).sum(),
          'Rendah', 'Putuskan: pertahankan atau gabungkan untuk analisis per wilayah')
catat_tim('Validitas', 'roles', 'Beberapa nilai dalam satu sel',
          df_tim['roles'].str.contains(',').sum(), 'Sedang', 'Pecah menjadi kolom biner per peran')
catat_tim('Validitas', '3D-model / gender', 'Atribut gabungan dan redundan (konsisten, tetapi dobel)',
          len(df_tim), 'Rendah', 'Pisahkan ukuran tubuh dari gender; pertahankan satu sumber')
catat_tim('Konsistensi', 'constellation_6_pull / constellation_7_pull', 'C7 > C6 pada banyak baris; makna kolom perlu dikonfirmasi',
          (df_tim['constellation_7_pull'] > df_tim['constellation_6_pull']).sum(), 'Sedang', 'Konfirmasi apakah C7 kumulatif sebelum dijumlahkan')
catat_tim('Validitas', 'is_standard_banner', 'Biner bertipe int64',
          len(df_tim), 'Rendah', 'Konversi ke bool pada pertemuan 5')
catat_tim('Validitas', 'pulled_count, c6_rate, num_banners', 'Pencilan IQR (belum tentu salah)',
          iqr_pencilan(df_tim['pulled_count']), 'Rendah', 'Pertahankan; gunakan median atau skala log')
catat_tim('Kelengkapan', 'seluruh kolom', 'Tidak ada NaN maupun placeholder (lolos)', 0, 'Rendah', 'Tidak ada tindakan')
catat_tim('Keunikan', 'seluruh kolom / name', 'Tidak ada duplikat persis maupun nyaris (lolos)', 0, 'Rendah', 'Tidak ada tindakan')

profil_tim = pd.DataFrame(temuan_tim)
profil_tim = profil_tim.sort_values(['keparahan', 'jumlah'], key=lambda c: c.map(urut) if c.name == 'keparahan' else -c)
profil_tim.to_csv('reports/profil_kualitas_data.csv', index=False)
print('Tersimpan di reports/profil_kualitas_data.csv:', profil_tim.shape)
profil_tim

Tersimpan di reports/profil_kualitas_data.csv: (11, 7)


,dimensi,kolom,masalah,jumlah,persen,keparahan,rencana
0,Kelengkapan,pulled_count dan turunannya,Dugaan bias pengumpulan (MNAR di level sumber ...,0,0.00,Tinggi,Tanyakan metode pengumpulan ke pemilik data; t...
4,Validitas,roles,Beberapa nilai dalam satu sel,103,100.00,Sedang,Pecah menjadi kolom biner per peran
6,Konsistensi,constellation_6_pull / constellation_7_pull,C7 > C6 pada banyak baris; makna kolom perlu d...,50,48.54,Sedang,Konfirmasi apakah C7 kumulatif sebelum dijumla...
2,Konsistensi,region,Salah eja Extra-Terresterial,1,0.97,Sedang,Koreksi menjadi Extra-Terrestrial; catat di cl...
5,Validitas,3D-model / gender,"Atribut gabungan dan redundan (konsisten, teta...",103,100.00,Rendah,Pisahkan ukuran tubuh dari gender; pertahankan...
7,Validitas,is_standard_banner,Biner bertipe int64,103,100.00,Rendah,Konversi ke bool pada pertemuan 5
1,Akurasi,pulled_count,"Tidak sama dengan jumlah C0-C7 (selisih 1-3, d...",9,8.74,Rendah,Konfirmasi cara hitung; jangan diubah sebelum ...
8,Validitas,"pulled_count, c6_rate, num_banners",Pencilan IQR (belum tentu salah),9,8.74,Rendah,Pertahankan; gunakan median atau skala log
3,Konsistensi,region,Kategori sangat jarang (<= 2 baris),3,2.91,Rendah,Putuskan: pertahankan atau gabungkan untuk ana...
9,Kelengkapan,seluruh kolom,Tidak ada NaN maupun placeholder (lolos),0,0.00,Rendah,Tidak ada tindakan


### Daftar periksa Bagian 10

- [x] Seluruh sembilan pemeriksaan dijalankan pada data tim
- [x] Setiap kolom yang punya nilai kosong di atas 5% sudah dinilai MCAR, MAR, atau MNAR, **beserta alasannya** (tidak ada kolom kosong; dugaan MNAR pada level sumber data dicatat)
- [x] Minimal lima aturan validitas ditulis sendiri berdasarkan pengetahuan tentang data tim (ada delapan aturan rentang/domain)
- [x] Minimal satu aturan hubungan antarkolom diperiksa (ada dua: jumlah C0-C7 dan rumus turunan)
- [x] Profil kualitas data tersimpan di `reports/`
- [ ] Notebook di-*commit* ke repositori tim dengan pesan yang bermakna *(dikerjakan tim)*

> **Ingat:** hari ini **tidak ada** yang diperbaiki. Perbaikan beserta justifikasinya adalah pekerjaan
> pertemuan 5. Yang diserahkan hari ini adalah diagnosis.

---
## Kalau Anda mentok

Ikuti urutan ini sebelum bertanya:

| Urutan | Yang dilakukan |
|---|---|
| 1 | Baca pesan galat **baris terakhirnya** — di situ nama galat dan penyebabnya |
| 2 | Jalankan ulang dari sel paling atas: *Runtime → Restart and run all* |
| 3 | Periksa kembali petunjuk yang muncul dari sel `cek(...)` |
| 4 | Tanyakan ke anggota tim; kemungkinan besar mereka menemui galat yang sama |
| 5 | Tanyakan ke asisten saat sesi berlangsung |
| 6 | Tulis di **forum tanya-jawab LMS**. Dosen menjawab setiap hari kerja. |

**Cara bertanya yang membuat jawaban datang lebih cepat:** sebutkan nomor bagian, tempelkan baris
terakhir pesan galat, dan sebutkan apa yang sudah Anda coba. Pertanyaan "kodenya error" tanpa keterangan
akan dibalas dengan pertanyaan balik, dan Anda kehilangan satu hari.

---
## Ringkasan pertemuan 4

1. **Lima dimensi:** kelengkapan, keunikan, konsistensi, validitas, akurasi. Periksa berurutan.
2. **Placeholder lolos dari `isna()`.** `-999` dan `N/A` terlihat seperti data sungguhan.
3. **MAR dapat dibuktikan; MCAR dan MNAR tidak.** Membedakannya butuh pengetahuan tentang cara data dikumpulkan.
4. **Duplikat nyaris lebih berbahaya daripada duplikat persis**, karena `duplicated()` tidak menangkapnya.
5. **Spasi tersembunyi menciptakan entitas hantu** dan diam-diam merusak hasil agregasi.
5b. **Konversi tanggal bisa salah tanpa satu pun galat.** Selalu periksa rentang dan bulan hasilnya.
6. **Satu cacat menyembunyikan cacat lain.** Diagnosis diulang setelah pembersihan.
7. **Hari ini mendiagnosis, belum memperbaiki.**
8. **Label `clean` bukan bukti.** Dataset yang lolos pemeriksaan tetap menyisakan temuan halus (salah eja, kolom majemuk, selisih pembulatan).

### Yang dikerjakan sebelum pertemuan 5

- Selesaikan Bagian 10 bersama tim dan *commit* ke repositori.
- Kerjakan kuis formatif pertemuan 4 di LMS (boleh diulang sebanyak yang Anda mau).
- Baca Bagian 1 workbook pertemuan 5 agar sesi berikutnya bisa langsung praktik.